<a href="https://colab.research.google.com/github/nattagon256312-afk/hello-chilli/blob/main/chili_Train_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import tensorflow as tf
import os

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!unzip -q "/content/drive/MyDrive/Full_chili.zip" -d "/content/dataset"

In [ ]:
import os
import shutil


base_dir = "/content/dataset/Chilli Plant Diseases Dataset(Augmented)/Chilli Plant Diseases Dataset"
train_dir = os.path.join(base_dir, "train")
valid_dir = os.path.join(base_dir, "valid")


if os.path.exists(valid_dir):
    for class_name in os.listdir(valid_dir):
        valid_class_path = os.path.join(valid_dir, class_name)
        train_class_path = os.path.join(train_dir, class_name)


        if os.path.isdir(valid_class_path):
            os.makedirs(train_class_path, exist_ok=True)


            for file_name in os.listdir(valid_class_path):
                src_file = os.path.join(valid_class_path, file_name)
                dst_file = os.path.join(train_class_path, file_name)
                shutil.move(src_file, dst_file)



In [ ]:

train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    train_dir,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=(160, 160),
    batch_size=32,
)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    train_dir,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(160, 160),
    batch_size=32,
)


test_dir = os.path.join(base_dir, "test")
test_ds = tf.keras.preprocessing.image_dataset_from_directory(
    test_dir, image_size=(160, 160), batch_size=32, shuffle=False
)

In [ ]:
data_amentation = tf.keras.Sequential(
    [
        tf.keras.layers.RandomFlip("horizontal_and_vertical"),
        tf.keras.layers.RandomRotation(0.2),
        tf.keras.layers.RandomZoom(0.2),
    ]
)


In [ ]:
base_model = tf.keras.applications.MobileNetV2(input_shape=(160,160,3), include_top=False, weights="imagenet", alpha=0.35)
base_model.trainable = False

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(160,160,3)),
    data_amentation,
    tf.keras.layers.Lambda(
        tf.keras.applications.mobilenet_v2.preprocess_input
    ),
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(8, activation="softmax"),
])

model.compile(optimizer=tf.keras.optimizers.RMSprop(learning_rate=0.0001,),loss="sparse_categorical_crossentropy",metrics=["accuracy"])


In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)


In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)

In [ ]:
import numpy as np
from sklearn.utils.class_weight import compute_class_weight


y_train = np.concatenate([y for x, y in train_ds], axis=0)

class_weights = compute_class_weight(
    class_weight="balanced", classes=np.unique(y_train), y=y_train
)

class_weight_dict = dict(enumerate(class_weights))

print("Class Weights ที่คำนวณได้:")
for idx, weight in class_weight_dict.items():
    print(f"คลาส {idx} (index {idx}): Weight = {weight:.4f}")

In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=16,
    class_weight=class_weight_dict,
    callbacks=[early_stop],
)

In [ ]:
test_loss, test_acc = model.evaluate(test_ds)
print(f"Test accuracy: {test_acc:.4f}")

In [ ]:
model.save("chilli_model.keras")